In [0]:
from pyspark.sql import functions as F

staging_customers = spark.table("brazilian_ecommerce.prod.staging_customers")
staging_orders = spark.table("brazilian_ecommerce.prod.staging_orders")
staging_order_payments = spark.table("brazilian_ecommerce.prod.staging_order_payments")
staging_order_reviews = spark.table("brazilian_ecommerce.prod.staging_order_reviews")

# --- orders joined to customer_unique_id (bridge customer_id -> customer_unique_id) ---
orders_with_unique_customer = (
    staging_orders
    .join(staging_customers.select("customer_id", "customer_unique_id"), on="customer_id", how="left")
)

# --- total spend per order (sum of payments), then rolled up to customer_unique_id ---
order_totals = (
    staging_order_payments
    .groupBy("order_id")
    .agg(F.sum("payment_value").cast("decimal(10,2)").alias("order_total_value"))
)

spend_and_dates = (
    orders_with_unique_customer
    .join(order_totals, on="order_id", how="left")
    .groupBy("customer_unique_id")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("order_total_value").cast("decimal(10,2)").alias("total_spend"),
        F.min(F.to_date("order_purchase_timestamp")).alias("first_purchase_date"),
        F.max(F.to_date("order_purchase_timestamp")).alias("last_purchase_date"),
    )
)

# --- avg review score given, via order_id -> customer_unique_id ---
review_scores = (
    orders_with_unique_customer.select("order_id", "customer_unique_id")
    .join(staging_order_reviews.select("order_id", "review_score"), on="order_id", how="left")
    .groupBy("customer_unique_id")
    .agg(F.avg("review_score").cast("decimal(3,2)").alias("avg_review_score_given"))
)

# --- combine everything ---
gold_customer_summary = (
    spend_and_dates
    .join(review_scores, on="customer_unique_id", how="left")
    .withColumn("_loaded_at", F.current_timestamp())
    .orderBy(F.desc("total_spend"))
)

full_table_name = "brazilian_ecommerce.prod.gold_customer_summary"

(
    gold_customer_summary.write
    .mode("overwrite")
    .option("comment", "gold table: one row per unique customer, with order volume, lifetime spend, review behavior and purchase dates")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ALTER COLUMN customer_unique_id SET NOT NULL
""")

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ADD CONSTRAINT pk_gold_customer_summary PRIMARY KEY (customer_unique_id)
""")

print(f"Loaded {gold_customer_summary.count()} rows into {full_table_name}")